# DHE (Deep Hash Embedding) — Criteo Benchmark
**Paper:** "Learning to Embed Categorical Features without Embedding Tables for Recommendation" — Google, KDD 2021

## Dataset: `nmpogg/criteo-cleaned-v2`
- Dữ liệu đã được tiền xử lý sẵn: median impute cho numeric, `log1p` transform, fill null cho categorical
- Categorical columns đã được mã hóa thành integer IDs (từ `tame_ivs_dictionary.json`)
- `tame_vocab_sizes.json` chứa vocab size của từng categorical feature
- Split sẵn: `train.parquet`, `validation.parquet`, `test.parquet`

## Architecture Overview
- **DHE**: Thay thế lookup table bằng một MLP nhỏ ánh xạ hash-code → embedding vector 64 chiều
- **DeepFM & DCN**: 13 integer features → đã `log1p` sẵn → concat trực tiếp; 26 cat features → DHE → 64-dim
- **DLRM**: 13 integer features → MLP → 64-dim; 26 cat features → DHE → 64-dim
- **Loss**: BCEWithLogitsLoss (model output raw logits, KHÔNG sigmoid bên trong)
- **Metric**: AUC-ROC

In [2]:
# ============================================================
# Cell 1 — Install & Imports
# ============================================================
import os, math, time, warnings, gc
warnings.filterwarnings('ignore')

try:
    import psutil
except ImportError:
    psutil = None

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

print(f'PyTorch: {torch.__version__}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')

PyTorch: 2.10.0+cu128
Device : cuda


In [3]:
# ============================================================
# Cell 2 — CONFIG
# ============================================================
CFG = {
    # --- Data ---
    # Dataset: https://www.kaggle.com/datasets/nmpogg/criteo-cleaned-v2
    'data_dir'  : '/kaggle/input/datasets/nmpogg/criteo-cleaned-v2',

    # --- DHE ---
    'embed_dim'   : 64,         # chiều output embedding
    'dhe_num_hash': 1024,       # K: số hash functions
    'dhe_hidden'  : [512, 256], # hidden layers của DHE MLP

    # --- Integer features (cho DLRM) ---
    'int_mlp_dims': [64, 64],   # MLP embed integer → 64 dim

    # --- Training ---
    'batch_size'  : 4096,
    'epochs'      : 5,
    'lr'          : 1e-3,
    'weight_decay': 1e-5,

    # --- Profiling / Resource metrics ---
    'profile_params': True,
    'profile_flops' : True,
    'measure_resources': True,
    'latency_warmup_batches': 5,

    # --- Models to run ---
    'deepfm_hidden'   : [400, 400, 400],
    'dcn_cross_layers': 3,
    'dcn_hidden'      : [512, 512],
    'dlrm_bottom_mlp' : [256, 128, 64],
    'dlrm_top_mlp'    : [512, 256, 128],
}

DENSE_COLS = [f'I{i}' for i in range(1, 14)]
CAT_COLS   = [f'C{i}' for i in range(1, 27)]
LABEL_COL  = 'label'
print('Config loaded.')

Config loaded.


In [4]:
# ============================================================
# Cell 3 — Load Data
# I1-I13 : đã log1p sẵn, dùng trực tiếp
# C1-C26 : string (vd: 'ad3062eb', 'unknownc1'), cần LabelEncoder
# ============================================================
data_dir = CFG['data_dir']

print('Loading parquet splits...')
train_df = pd.read_parquet(os.path.join(data_dir, 'train.parquet'))
val_df   = pd.read_parquet(os.path.join(data_dir, 'validation.parquet'))
test_df  = pd.read_parquet(os.path.join(data_dir, 'test.parquet'))

print(f'Train:      {len(train_df):>10,} rows')
print(f'Validation: {len(val_df):>10,} rows')
print(f'Test:       {len(test_df):>10,} rows')
print(train_df.head(2))


Loading parquet splits...
Train:      36,665,662 rows
Validation:  4,583,562 rows
Test:        4,591,393 rows
   label        I1        I2        I3        I4        I5        I6  \
0      0  0.693147  3.295837  1.098612  1.098612  6.645091  3.526361   
1      1  1.791759  0.000000  1.386294  2.197225  1.098612  1.945910   

         I7        I8       I9  ...       C17       C18         C19  \
0  0.000000  2.484907  2.70805  ...  1e88c74f  065917ca  unknownc19   
1  3.610918  3.367296  4.51086  ...  27c07bd6  88f30ecb  unknownc19   

          C20         C21         C22       C23         C24         C25  \
0  unknownc20    3f96e661  unknownc22  32c7478e    128f60b8  unknownc25   
1  unknownc20  unknownc21  unknownc22  423fab69  unknownc24  unknownc25   

          C26  
0  unknownc26  
1  unknownc26  

[2 rows x 40 columns]


In [5]:
# ============================================================
# Cell 4 — LabelEncode categorical + cast dtypes
# C1-C26 là string → encode thành integer index cho DHE
# Fit trên train, transform val/test
# Val/test không có unseen values vì NaN đã fill thành 'unknownCx' rồi
# ============================================================
vocab_sizes = []

for col in CAT_COLS:
    le = LabelEncoder()
    le.fit(train_df[col].astype(str))

    train_df[col] = le.transform(train_df[col].astype(str))

    # Val/test: nếu có giá trị chưa thấy trong train → map về 0
    def safe_transform(series, le=le):
        s = series.astype(str)
        known = set(le.classes_)
        s = s.where(s.isin(known), other=le.classes_[0])
        return le.transform(s)

    val_df[col]  = safe_transform(val_df[col])
    test_df[col] = safe_transform(test_df[col])

    vocab_sizes.append(len(le.classes_))

# Cast dtypes
for df in [train_df, val_df, test_df]:
    df[DENSE_COLS] = df[DENSE_COLS].astype(np.float32)
    df[CAT_COLS]   = df[CAT_COLS].astype(np.int64)
    df[LABEL_COL]  = df[LABEL_COL].astype(np.float32)

print('Vocab sizes (first 5):', vocab_sizes[:5])
print('Total cat features   :', len(vocab_sizes))
print('\nDense sample (đã log1p):')
print(train_df[DENSE_COLS].head(2))
print('\nCat encoded:')
print(train_df[CAT_COLS].head(2))


Vocab sizes (first 5): [1460, 577, 8378273, 1884135, 305]
Total cat features   : 26

Dense sample (đã log1p):
         I1        I2        I3        I4        I5        I6        I7  \
0  0.693147  3.295837  1.098612  1.098612  6.645091  3.526361  0.000000   
1  1.791759  0.000000  1.386294  2.197225  1.098612  1.945910  3.610918   

         I8        I9       I10       I11  I12       I13  
0  2.484907  2.708050  0.693147  0.000000  0.0  1.098612  
1  3.367296  4.510859  1.098612  2.197225  0.0  1.945910  

Cat encoded:
    C1   C2       C3       C4  C5  C6    C7  C8  C9    C10  ...  C17   C18  \
0   31  129  4213993   905630  43   9  3074  24   2  25513  ...    1   124   
1  629  129  8378272  1884134  43   9  6908  24   2  33030  ...    3  2936   

    C19  C20      C21  C22  C23     C24  C25     C26  
0  2167    3  1462064   17    2   18522  104  133142  
1  2167    3  5893093   17    4  258694  104  133142  

[2 rows x 26 columns]


In [6]:
# ============================================================
# Cell 6 — Dataset
# ============================================================
class CriteoDataset(Dataset):
    def __init__(self, df, dense_cols, cat_cols, label_col):
        self.dense = torch.tensor(df[dense_cols].values, dtype=torch.float32)
        self.cat   = torch.tensor(df[cat_cols].values,   dtype=torch.long)
        self.label = torch.tensor(df[label_col].values,  dtype=torch.float32)

    def __len__(self):
        return len(self.label)

    def __getitem__(self, idx):
        return self.dense[idx], self.cat[idx], self.label[idx]

train_ds = CriteoDataset(train_df, DENSE_COLS, CAT_COLS, LABEL_COL)
val_ds   = CriteoDataset(val_df,   DENSE_COLS, CAT_COLS, LABEL_COL)
test_ds  = CriteoDataset(test_df,  DENSE_COLS, CAT_COLS, LABEL_COL)

train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=CFG['batch_size']*2, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=CFG['batch_size']*2, shuffle=False, num_workers=2, pin_memory=True)
print('DataLoaders ready.')

# ============================================================
# Cell 7 — DHE Module
# ============================================================
# Nguồn: "Learning to Embed Categorical Features without Embedding Tables" KDD 2021
# Ý tưởng: với mỗi category ID, tạo hash code K chiều bằng K hàm hash độc lập
# rồi đưa qua MLP để ra embedding 64 chiều.
# Không dùng bảng lookup → tiết kiệm memory với vocab lớn, tổng quát hoá tốt hơn.

class DHEEncoder(nn.Module):
    """
    Deep Hash Embedding cho một feature với vocab_size tùy ý.
    Input : (B,) LongTensor các category ID
    Output: (B, embed_dim) FloatTensor
    """
    def __init__(self, vocab_size: int, embed_dim: int, num_hashes: int = 1024, hidden_dims=(512, 256)):
        super().__init__()
        self.vocab_size = vocab_size
        self.num_hashes = num_hashes
        self.embed_dim  = embed_dim

        # Random prime numbers dùng làm hash seeds — cố định để stable
        # Dùng CRC-style universal hashing: h_i(x) = ((a_i * x + b_i) mod p) mod num_hashes
        # Để tránh collisions, dùng large prime p
        torch.manual_seed(42)
        self.register_buffer('a_vec', torch.randint(1, 2**31 - 1, (num_hashes,), dtype=torch.long))
        self.register_buffer('b_vec', torch.randint(0, 2**31 - 1, (num_hashes,), dtype=torch.long))
        self.p = 2**31 - 1  # Mersenne prime

        # MLP: num_hashes → embed_dim
        # Input sau hash: binary ±1 encoding
        layers = []
        in_d = num_hashes
        for h in hidden_dims:
            layers += [nn.Linear(in_d, h), nn.LayerNorm(h), nn.GELU()]
            in_d = h
        layers.append(nn.Linear(in_d, embed_dim))
        self.mlp = nn.Sequential(*layers)

    def _hash_encode(self, ids: torch.Tensor) -> torch.Tensor:
        """
        ids: (B,) long
        returns: (B, K) float in {-1, +1}
        """
        B = ids.size(0)
        # (B, K)
        ids_exp = ids.unsqueeze(1).expand(B, self.num_hashes)        # (B, K)
        # Universal hash
        h = ((self.a_vec * ids_exp + self.b_vec) % self.p) % 2      # (B, K) in {0,1}
        return h.float() * 2 - 1  # map to {-1, +1}

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        code = self._hash_encode(ids)  # (B, K)
        return self.mlp(code)          # (B, embed_dim)


class DHELayer(nn.Module):
    """
    Wrapper áp dụng DHE cho tất cả 26 cat features song song.
    Input : (B, 26) LongTensor
    Output: (B, 26, embed_dim)
    """
    def __init__(self, vocab_sizes, embed_dim, num_hashes, hidden_dims):
        super().__init__()
        self.encoders = nn.ModuleList([
            DHEEncoder(v, embed_dim, num_hashes, hidden_dims)
            for v in vocab_sizes
        ])
        self.embed_dim = embed_dim

    def forward(self, cat_x: torch.Tensor) -> torch.Tensor:
        # cat_x: (B, 26)
        embs = [enc(cat_x[:, i]) for i, enc in enumerate(self.encoders)]  # list of (B, D)
        return torch.stack(embs, dim=1)  # (B, 26, D)

print('DHE module defined.')

DataLoaders ready.
DHE module defined.


In [7]:
# ============================================================
# Cell 8 — Numeric Feature Processors
# Data đã log1p sẵn → chỉ cần LayerNorm để ổn định training
# DenseEmbedMLP chỉ dùng cho DLRM: scalar → 64-dim vector
# ============================================================

class DenseNormalizer(nn.Module):
    """
    Data đã log1p sẵn từ preprocessing.
    Chỉ cần LayerNorm để re-scale trước khi vào model.
    """
    def __init__(self, num_features):
        super().__init__()
        self.norm = nn.LayerNorm(num_features)

    def forward(self, x):
        return self.norm(x)


class DenseEmbedMLP(nn.Module):
    """
    Chỉ dùng cho DLRM: embed mỗi scalar dense feature → vector 64-dim.
    Input : (B, 13)
    Output: (B, 13, embed_dim)
    """
    def __init__(self, num_dense, embed_dim, hidden_dims):
        super().__init__()
        layers = []
        in_d = 1
        for h in hidden_dims:
            layers += [nn.Linear(in_d, h), nn.LayerNorm(h), nn.GELU()]
            in_d = h
        layers.append(nn.Linear(in_d, embed_dim))
        self.mlp = nn.Sequential(*layers)
        self.norm = nn.LayerNorm(1)
        self.num_dense = num_dense

    def forward(self, x):
        out = []
        for i in range(self.num_dense):
            xi = x[:, i:i+1]       # (B, 1)
            xi = self.norm(xi)
            out.append(self.mlp(xi))
        return torch.stack(out, dim=1)  # (B, 13, embed_dim)

print('Numeric processors defined.')


Numeric processors defined.


In [8]:
# ============================================================
# Cell 9 — DeepFM with DHE  (FIXED: BCEWithLogitsLoss, no sigmoid in model)
# ============================================================
class DeepFM_DHE(nn.Module):
    """
    DeepFM với DHE embedding cho categorical features.
    Input dense  : (B, 13) → log1p normalize → concat trực tiếp
    Input sparse : (B, 26) → DHE → (B, 26, 64)

    FIX so với bản gốc:
    - Bỏ sigmoid() ở output, dùng BCEWithLogitsLoss
    - FM terms được normalize trước khi cộng để tránh gradient explosion
    - LayerNorm trên FM output
    """
    def __init__(self, num_dense, vocab_sizes, embed_dim, hidden_dims,
                 dhe_num_hashes, dhe_hidden):
        super().__init__()
        self.num_dense = num_dense
        self.num_sparse = len(vocab_sizes)
        self.embed_dim = embed_dim

        # Dense normalization
        self.dense_norm = DenseNormalizer(num_dense)

        # DHE for cat features
        self.dhe = DHELayer(vocab_sizes, embed_dim, dhe_num_hashes, dhe_hidden)

        # FM order-1: linear weights cho sparse (scalar per feature)
        # Không dùng Embedding cho FM-1 vì DHE đã encode, ta dùng linear projection
        self.fm1_sparse_proj = nn.Linear(embed_dim, 1, bias=False)  # shared
        self.fm1_dense_proj  = nn.Linear(num_dense, 1)

        # FM order-2 normalization
        self.fm2_norm = nn.LayerNorm(1)

        # Deep component
        deep_in = num_dense + self.num_sparse * embed_dim
        layers = []
        in_d = deep_in
        for dim in hidden_dims:
            layers += [nn.Linear(in_d, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(0.1)]
            in_d = dim
        layers.append(nn.Linear(in_d, 1))
        self.deep = nn.Sequential(*layers)

    def forward(self, dense_x, sparse_x):
        # Normalize dense
        dense_norm = self.dense_norm(dense_x)  # (B, 13)

        # DHE embeddings: (B, 26, 64)
        emb = self.dhe(sparse_x)  # (B, 26, 64)

        # --- FM Order-1 ---
        # Sparse: sum over features after linear proj
        fm1_sparse = self.fm1_sparse_proj(emb).squeeze(-1).sum(dim=1, keepdim=True)  # (B,1)
        fm1_dense  = self.fm1_dense_proj(dense_norm)                                  # (B,1)
        fm1 = fm1_sparse + fm1_dense

        # --- FM Order-2 ---
        # Chỉ tính trên sparse embeddings
        sum_emb    = emb.sum(dim=1)         # (B, 64)
        sum_sq_emb = sum_emb ** 2
        sq_sum_emb = (emb ** 2).sum(dim=1)
        fm2 = 0.5 * (sum_sq_emb - sq_sum_emb).sum(dim=1, keepdim=True)  # (B,1)
        fm2 = self.fm2_norm(fm2)  # normalize để tránh scale mismatch

        # --- Deep ---
        deep_in = torch.cat([dense_norm, emb.view(dense_x.size(0), -1)], dim=1)
        deep_out = self.deep(deep_in)  # (B,1)

        # Raw logit (NO sigmoid)
        return fm1 + fm2 + deep_out  # (B,1)

print('DeepFM_DHE defined.')

DeepFM_DHE defined.


In [9]:
# ============================================================
# Cell 10 — DCN with DHE  (FIXED)
# ============================================================
class CrossNetwork(nn.Module):
    """Cross Network — DCN-V1 style."""
    def __init__(self, input_dim, num_layers):
        super().__init__()
        self.num_layers = num_layers
        self.w = nn.ParameterList([nn.Parameter(torch.empty(input_dim)) for _ in range(num_layers)])
        self.b = nn.ParameterList([nn.Parameter(torch.zeros(input_dim)) for _ in range(num_layers)])
        for w in self.w:
            nn.init.xavier_uniform_(w.unsqueeze(0))

    def forward(self, x0):
        xl = x0
        for i in range(self.num_layers):
            xlw = (xl * self.w[i]).sum(dim=1, keepdim=True)  # (B,1)
            xl  = x0 * xlw + self.b[i] + xl
        return xl


class DCN_DHE(nn.Module):
    """
    DCN với DHE embedding.
    Dense: log1p normalize (không embed thành 64 chiều)
    Sparse: DHE → 64 chiều
    Input dim = 13 + 26*64 = 1677

    FIX: bỏ sigmoid trong model, dùng BCEWithLogitsLoss
    """
    def __init__(self, num_dense, vocab_sizes, embed_dim, cross_layers, hidden_dims,
                 dhe_num_hashes, dhe_hidden):
        super().__init__()
        self.num_dense  = num_dense
        self.num_sparse = len(vocab_sizes)
        self.embed_dim  = embed_dim

        self.dense_norm = DenseNormalizer(num_dense)
        self.dhe = DHELayer(vocab_sizes, embed_dim, dhe_num_hashes, dhe_hidden)

        input_dim = num_dense + self.num_sparse * embed_dim  # 13 + 26*64 = 1677
        self.cross_net = CrossNetwork(input_dim, cross_layers)

        deep_layers = []
        in_d = input_dim
        for dim in hidden_dims:
            deep_layers += [nn.Linear(in_d, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(0.1)]
            in_d = dim
        self.deep_net = nn.Sequential(*deep_layers)

        # Output: cross + deep → 1
        self.fc = nn.Linear(input_dim + in_d, 1)

    def forward(self, dense_x, sparse_x):
        dense_norm = self.dense_norm(dense_x)             # (B, 13)
        emb = self.dhe(sparse_x)                          # (B, 26, 64)
        emb_flat = emb.view(dense_x.size(0), -1)          # (B, 1664)

        x0 = torch.cat([dense_norm, emb_flat], dim=1)     # (B, 1677)

        cross_out = self.cross_net(x0)                    # (B, 1677)
        deep_out  = self.deep_net(x0)                     # (B, last_dim)

        combined = torch.cat([cross_out, deep_out], dim=1)
        return self.fc(combined)  # raw logit, NO sigmoid

print('DCN_DHE defined.')

DCN_DHE defined.


In [10]:
# ============================================================
# Cell 11 — DLRM with DHE
# ============================================================
class DLRM_DHE(nn.Module):
    """
    DLRM với DHE embedding.
    Dense features: MLP → 64-dim (Bottom MLP alternative với DenseEmbedMLP)
    Sparse features: DHE → 64-dim
    Tất cả features → 64-dim → Dot-product interactions → Top MLP
    """
    def __init__(self, num_dense, vocab_sizes, embed_dim,
                 dense_embed_hidden, top_mlp_dims,
                 dhe_num_hashes, dhe_hidden):
        super().__init__()
        self.num_dense  = num_dense
        self.num_sparse = len(vocab_sizes)
        self.embed_dim  = embed_dim

        # Dense: 13 scalar features → 13 vectors of dim 64
        self.dense_embed = DenseEmbedMLP(num_dense, embed_dim, dense_embed_hidden)

        # Sparse: DHE
        self.dhe = DHELayer(vocab_sizes, embed_dim, dhe_num_hashes, dhe_hidden)

        # Số features: 13 dense + 26 sparse = 39
        num_all = num_dense + self.num_sparse  # 39
        # Số dot-product pairs (upper triangle, offset=1)
        num_interactions = (num_all * (num_all - 1)) // 2
        # Top MLP input: 1 dense vec (embed_dim) + interactions
        # Theo paper DLRM, concat embed_dim (từ bottom MLP) + interactions
        # Ở đây ta dùng mean của dense embeds thay cho 1 bottom MLP
        top_in = embed_dim + num_interactions

        top_layers = []
        in_d = top_in
        for dim in top_mlp_dims:
            top_layers += [nn.Linear(in_d, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(0.1)]
            in_d = dim
        top_layers.append(nn.Linear(in_d, 1))
        self.top_mlp = nn.Sequential(*top_layers)

    def forward(self, dense_x, sparse_x):
        # Dense: (B, 13) → (B, 13, 64)
        dense_emb = self.dense_embed(dense_x)   # (B, 13, 64)
        # Sparse: (B, 26) → (B, 26, 64)
        sparse_emb = self.dhe(sparse_x)          # (B, 26, 64)

        # Concat all: (B, 39, 64)
        all_emb = torch.cat([dense_emb, sparse_emb], dim=1)  # (B, 39, 64)

        # Dot-product interactions
        Z = torch.bmm(all_emb, all_emb.transpose(1, 2))  # (B, 39, 39)
        n = all_emb.size(1)
        rows, cols = torch.triu_indices(n, n, offset=1)
        interactions = Z[:, rows, cols]  # (B, num_interactions)

        # Dùng mean của dense embeds làm "dense representative"
        dense_rep = dense_emb.mean(dim=1)  # (B, 64)

        concat = torch.cat([dense_rep, interactions], dim=1)  # (B, 64+num_int)
        return self.top_mlp(concat)  # raw logit

print('DLRM_DHE defined.')

DLRM_DHE defined.


In [11]:
# ============================================================
# Cell 12 — Profiling Utilities: Params / FLOPs / RAM / VRAM / Latency
# ============================================================
def _safe_gb(value):
    return value / (1024 ** 3)


def current_ram_gb():
    if psutil is None:
        return float('nan')
    return _safe_gb(psutil.virtual_memory().used)


def current_vram_gb(device):
    if device.type != 'cuda':
        return 0.0
    return _safe_gb(torch.cuda.memory_allocated(device))


def peak_vram_gb(device):
    if device.type != 'cuda':
        return 0.0
    return _safe_gb(torch.cuda.max_memory_allocated(device))


def print_parameter_report(model, top_k=40):
    rows = []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        rows.append({
            'name': name,
            'shape': tuple(p.shape),
            'params': p.numel(),
            'memory_mb': p.numel() * p.element_size() / (1024 ** 2),
        })

    total_params = sum(r['params'] for r in rows)
    total_param_bytes = sum(p.numel() * p.element_size() for p in model.parameters() if p.requires_grad)
    grad_bytes = total_param_bytes
    adam_state_bytes = total_param_bytes * 2  # Adam/AdamW stores m and v states.
    train_state_bytes = total_param_bytes + grad_bytes + adam_state_bytes

    print(f'Trainable params: {total_params:,}')
    print(f'Parameter memory: {_safe_gb(total_param_bytes):.3f} GB')
    print(f'Grad memory     : {_safe_gb(grad_bytes):.3f} GB')
    print(f'Adam states est.: {_safe_gb(adam_state_bytes):.3f} GB')
    print(f'Train-state est.: {_safe_gb(train_state_bytes):.3f} GB (params + grads + Adam states, no activations)')

    by_root = {}
    for r in rows:
        root = r['name'].split('.')[0]
        by_root[root] = by_root.get(root, 0) + r['params']

    print('\nParams by top module:')
    for root, count in sorted(by_root.items(), key=lambda x: x[1], reverse=True):
        print(f'  {root:<18} {count:>15,}')

    print(f'\nLargest parameter tensors (top {min(top_k, len(rows))}):')
    for r in sorted(rows, key=lambda x: x['params'], reverse=True)[:top_k]:
        print(f"  {r['name']:<55} {str(r['shape']):<22} {r['params']:>12,}  {r['memory_mb']:>9.2f} MB")

    return {
        'trainable_params': total_params,
        'parameter_gb': _safe_gb(total_param_bytes),
        'grad_gb': _safe_gb(grad_bytes),
        'adam_state_gb': _safe_gb(adam_state_bytes),
        'train_state_est_gb': _safe_gb(train_state_bytes),
    }


def profile_model_flops(model, device, num_dense, num_sparse, batch_size=1):
    try:
        from torch.utils.flop_counter import FlopCounterMode
    except Exception as exc:
        print(f'FLOP counter unavailable: {exc}')
        return None

    was_training = model.training
    model.eval()
    dummy_dense = torch.randn(batch_size, num_dense, device=device)
    dummy_cat = torch.zeros(batch_size, num_sparse, dtype=torch.long, device=device)

    print(f'\nFLOPs profile with dummy batch_size={batch_size}:')
    try:
        try:
            flop_counter = FlopCounterMode(model, display=True)
        except TypeError:
            flop_counter = FlopCounterMode(display=True)
        with torch.no_grad(), flop_counter:
            _ = model(dummy_dense, dummy_cat)
        if device.type == 'cuda':
            torch.cuda.synchronize()
    except Exception as exc:
        print(f'FLOP profiling failed: {exc}')
        return None
    finally:
        model.train(was_training)
        del dummy_dense, dummy_cat
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    return True


def profile_model_once(model_name, model, cfg, device, num_dense, num_sparse):
    print('\n' + '-' * 60)
    print(f'Profiling {model_name}-DHE')
    print('-' * 60)

    param_metrics = None
    if cfg.get('profile_params', True):
        param_metrics = print_parameter_report(model)

    if cfg.get('profile_flops', True):
        profile_model_flops(model, device, num_dense, num_sparse, batch_size=1)

    return param_metrics


def summarize_resource_samples(ram_samples, vram_samples, device):
    if len(ram_samples) == 0:
        avg_ram = float('nan')
    else:
        avg_ram = float(np.mean(ram_samples))

    avg_vram = float(np.mean(vram_samples)) if len(vram_samples) else 0.0
    return {
        'avg_ram_gb': avg_ram,
        'avg_vram_gb': avg_vram,
        'peak_vram_gb': peak_vram_gb(device),
    }


In [12]:
# ============================================================
# Cell 13 — Training Engine
# ============================================================
def train_one_epoch(model, loader, optimizer, criterion, device, scaler=None, measure_resources=True):
    model.train()
    total_loss = 0
    ram_samples, vram_samples = [], []

    for dense, cat, label in loader:
        dense, cat, label = dense.to(device), cat.to(device), label.to(device)
        optimizer.zero_grad()
        if scaler is not None:
            with torch.amp.autocast('cuda'):
                logit = model(dense, cat).squeeze(1)
                loss  = criterion(logit, label)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logit = model(dense, cat).squeeze(1)
            loss  = criterion(logit, label)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total_loss += loss.item() * label.size(0)

        if measure_resources:
            ram_samples.append(current_ram_gb())
            vram_samples.append(current_vram_gb(device))

    resource_metrics = summarize_resource_samples(ram_samples, vram_samples, device)
    return total_loss / len(loader.dataset), resource_metrics


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    for dense, cat, label in loader:
        dense, cat, label = dense.to(device), cat.to(device), label.to(device)
        logit = model(dense, cat).squeeze(1)
        loss  = criterion(logit, label)
        total_loss += loss.item() * label.size(0)
        all_preds.append(torch.sigmoid(logit).cpu().numpy())
        all_labels.append(label.cpu().numpy())
    preds  = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    auc    = roc_auc_score(labels, preds)
    return total_loss / len(loader.dataset), auc


@torch.no_grad()
def evaluate_with_latency(model, loader, criterion, device, warmup_batches=5, measure_resources=True):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    total_latency_ms, latency_batches, latency_samples = 0.0, 0, 0
    ram_samples, vram_samples = [], []

    use_cuda_timer = device.type == 'cuda'

    for i, (dense, cat, label) in enumerate(loader):
        dense, cat, label = dense.to(device), cat.to(device), label.to(device)
        batch_size = label.size(0)

        if use_cuda_timer:
            start_event = torch.cuda.Event(enable_timing=True)
            end_event = torch.cuda.Event(enable_timing=True)
            start_event.record()
            logit = model(dense, cat).squeeze(1)
            end_event.record()
            torch.cuda.synchronize()
            batch_latency_ms = start_event.elapsed_time(end_event)
        else:
            t0 = time.perf_counter()
            logit = model(dense, cat).squeeze(1)
            batch_latency_ms = (time.perf_counter() - t0) * 1000

        loss  = criterion(logit, label)
        total_loss += loss.item() * batch_size
        all_preds.append(torch.sigmoid(logit).cpu().numpy())
        all_labels.append(label.cpu().numpy())

        if i >= warmup_batches:
            total_latency_ms += batch_latency_ms
            latency_batches += 1
            latency_samples += batch_size

        if measure_resources:
            ram_samples.append(current_ram_gb())
            vram_samples.append(current_vram_gb(device))

    preds  = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    auc    = roc_auc_score(labels, preds)

    resource_metrics = summarize_resource_samples(ram_samples, vram_samples, device)
    if latency_batches > 0:
        resource_metrics.update({
            'avg_batch_latency_ms': total_latency_ms / latency_batches,
            'avg_sample_latency_ms': total_latency_ms / max(latency_samples, 1),
        })
    else:
        resource_metrics.update({
            'avg_batch_latency_ms': 0.0,
            'avg_sample_latency_ms': 0.0,
        })

    return total_loss / len(loader.dataset), auc, resource_metrics


def run_experiment(model_name, model, cfg, train_dl, val_dl, test_dl, device):
    sep = '='*60
    print(f'\n{sep}')
    print(f'  Model: {model_name}  |  Embedding: DHE')
    nparams = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Params: {nparams:,}')
    print(f'{sep}')

    model = model.to(device)
    profile_metrics = profile_model_once(
        model_name, model, cfg, device,
        num_dense=len(DENSE_COLS),
        num_sparse=len(CAT_COLS),
    )

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=cfg['lr'], steps_per_epoch=len(train_dl), epochs=cfg['epochs']
    )
    scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

    best_val_auc = 0
    results = []
    measure_resources = cfg.get('measure_resources', True)

    for epoch in range(1, cfg['epochs'] + 1):
        if device.type == 'cuda':
            torch.cuda.reset_peak_memory_stats(device)

        t0 = time.time()
        train_loss, train_resources = train_one_epoch(
            model, train_dl, optimizer, criterion, device, scaler,
            measure_resources=measure_resources,
        )
        scheduler.step()
        val_loss, val_auc = evaluate(model, val_dl, criterion, device)
        elapsed = time.time() - t0

        print(
            f'  Epoch {epoch}/{cfg["epochs"]} | '
            f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f} | {elapsed:.1f}s'
        )
        print(
            f'    Resources | RAM avg: {train_resources["avg_ram_gb"]:.2f} GB | '
            f'VRAM avg: {train_resources["avg_vram_gb"]:.2f} GB | '
            f'VRAM peak: {train_resources["peak_vram_gb"]:.2f} GB'
        )

        row = {
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_auc': val_auc,
            'epoch_seconds': elapsed,
            **train_resources,
        }
        if profile_metrics is not None:
            row.update(profile_metrics)
        results.append(row)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save(model.state_dict(), f'{model_name}_DHE_best.pt')

    # Test evaluation + latency
    model.load_state_dict(torch.load(f'{model_name}_DHE_best.pt', map_location=device))
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)

    test_loss, test_auc, test_metrics = evaluate_with_latency(
        model, test_dl, criterion, device,
        warmup_batches=cfg.get('latency_warmup_batches', 5),
        measure_resources=measure_resources,
    )
    print(f'\n  Test AUC: {test_auc:.4f}  |  Test Loss: {test_loss:.4f}')
    print(
        f'  Test resources | RAM avg: {test_metrics["avg_ram_gb"]:.2f} GB | '
        f'VRAM avg: {test_metrics["avg_vram_gb"]:.2f} GB | '
        f'VRAM peak: {test_metrics["peak_vram_gb"]:.2f} GB'
    )
    print(
        f'  Latency | Batch: {test_metrics["avg_batch_latency_ms"]:.2f} ms | '
        f'Sample: {test_metrics["avg_sample_latency_ms"]:.6f} ms'
    )

    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

    return results, test_auc

print('Training engine ready.')

Training engine ready.


In [13]:
# ============================================================
# Cell 14 — Run DeepFM + DHE
# ============================================================
deepfm_model = DeepFM_DHE(
    num_dense    = len(DENSE_COLS),
    vocab_sizes  = vocab_sizes,
    embed_dim    = CFG['embed_dim'],
    hidden_dims  = CFG['deepfm_hidden'],
    dhe_num_hashes = CFG['dhe_num_hash'],
    dhe_hidden   = CFG['dhe_hidden'],
)
deepfm_results, deepfm_test_auc = run_experiment(
    'DeepFM', deepfm_model, CFG, train_dl, val_dl, test_dl, DEVICE
)


  Model: DeepFM  |  Embedding: DHE
  Params: 18,521,819

------------------------------------------------------------
Profiling DeepFM-DHE
------------------------------------------------------------
Trainable params: 18,521,819
Parameter memory: 0.069 GB
Grad memory     : 0.069 GB
Adam states est.: 0.138 GB
Train-state est.: 0.276 GB (params + grads + Adam states, no activations)

Params by top module:
  dhe                     17,526,912
  deep                       994,801
  fm1_sparse_proj                 64
  dense_norm                      26
  fm1_dense_proj                  14
  fm2_norm                         2

Largest parameter tensors (top 40):
  deep.0.weight                                           (400, 1677)                 670,800       2.56 MB
  dhe.encoders.0.mlp.0.weight                             (512, 1024)                 524,288       2.00 MB
  dhe.encoders.1.mlp.0.weight                             (512, 1024)                 524,288       2.00 MB
  dhe.enc

In [14]:
# ============================================================
# Cell 15 — Run DCN + DHE
# ============================================================
dcn_model = DCN_DHE(
    num_dense    = len(DENSE_COLS),
    vocab_sizes  = vocab_sizes,
    embed_dim    = CFG['embed_dim'],
    cross_layers = CFG['dcn_cross_layers'],
    hidden_dims  = CFG['dcn_hidden'],
    dhe_num_hashes = CFG['dhe_num_hash'],
    dhe_hidden   = CFG['dhe_hidden'],
)
dcn_results, dcn_test_auc = run_experiment(
    'DCN', dcn_model, CFG, train_dl, val_dl, test_dl, DEVICE
)


  Model: DCN  |  Embedding: DHE
  Params: 18,663,030

------------------------------------------------------------
Profiling DCN-DHE
------------------------------------------------------------
Trainable params: 18,663,030
Parameter memory: 0.070 GB
Grad memory     : 0.070 GB
Adam states est.: 0.139 GB
Train-state est.: 0.278 GB (params + grads + Adam states, no activations)

Params by top module:
  dhe                     17,526,912
  deep_net                 1,123,840
  cross_net                   10,062
  fc                           2,190
  dense_norm                      26

Largest parameter tensors (top 40):
  deep_net.0.weight                                       (512, 1677)                 858,624       3.28 MB
  dhe.encoders.0.mlp.0.weight                             (512, 1024)                 524,288       2.00 MB
  dhe.encoders.1.mlp.0.weight                             (512, 1024)                 524,288       2.00 MB
  dhe.encoders.2.mlp.0.weight                       

In [ ]:
# ============================================================
# Cell 16 — Run DLRM + DHE
# ============================================================
dlrm_model = DLRM_DHE(
    num_dense          = len(DENSE_COLS),
    vocab_sizes        = vocab_sizes,
    embed_dim          = CFG['embed_dim'],
    dense_embed_hidden = CFG['int_mlp_dims'],
    top_mlp_dims       = CFG['dlrm_top_mlp'],
    dhe_num_hashes     = CFG['dhe_num_hash'],
    dhe_hidden         = CFG['dhe_hidden'],
)
dlrm_results, dlrm_test_auc = run_experiment(
    'DLRM', dlrm_model, CFG, train_dl, val_dl, test_dl, DEVICE
)


  Model: DLRM  |  Embedding: DHE
  Params: 18,114,435

------------------------------------------------------------
Profiling DLRM-DHE
------------------------------------------------------------
Trainable params: 18,114,435
Parameter memory: 0.067 GB
Grad memory     : 0.067 GB
Adam states est.: 0.135 GB
Train-state est.: 0.270 GB (params + grads + Adam states, no activations)

Params by top module:
  dhe                     17,526,912
  top_mlp                    578,817
  dense_embed                  8,706

Largest parameter tensors (top 40):
  dhe.encoders.0.mlp.0.weight                             (512, 1024)                 524,288       2.00 MB
  dhe.encoders.1.mlp.0.weight                             (512, 1024)                 524,288       2.00 MB
  dhe.encoders.2.mlp.0.weight                             (512, 1024)                 524,288       2.00 MB
  dhe.encoders.3.mlp.0.weight                             (512, 1024)                 524,288       2.00 MB
  dhe.encoders.4